In [ ]:
# ============================================================
#  SHORELINE BUFFER GENERATOR 
# ============================================================

# ── CELL 1: Mount Google Drive ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')


# ── CELL 2: Install Dependencies ────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'geopandas', 'shapely', 'pyproj', 'fiona', '--quiet'])
print('All packages ready!')


# ── CELL 3: Set Your Paths ───────────────────────────────────
import os

# === UPDATE THESE TWO PATHS ===
SHORELINE_SHP = '/content/drive/MyDrive/dem_output/Shoreline_2026.shp'
OUTPUT_FOLDER = '/content/drive/MyDrive/shoreline_buffer'
# ==============================

# Optional: path to a land polygon shapefile for line-type shorelines
# Leave as None if your shoreline is already a polygon
LAND_POLYGON_SHP = None
# Example: LAND_POLYGON_SHP = '/content/drive/MyDrive/GIS_Data/land_boundary.shp'

# Buffer distances in metres
BUFFER_DISTANCES = [500, 1000, 1500, 2000]

# Validate input
if not os.path.exists(SHORELINE_SHP):
    raise FileNotFoundError(
        f'Shoreline shapefile not found at:\n  {SHORELINE_SHP}\n'
        'Please update SHORELINE_SHP to the correct Google Drive path.'
    )

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f'Input  : {SHORELINE_SHP}')
print(f'Output : {OUTPUT_FOLDER}')
print(f'Buffers: {BUFFER_DISTANCES} m')


# ── CELL 4: Load Shoreline & Reproject ──────────────────────
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

print('Loading shoreline shapefile...')
shoreline = gpd.read_file(SHORELINE_SHP)
print(f'  CRS      : {shoreline.crs}')
print(f'  Features : {len(shoreline)}')
print(f'  Geometry : {list(shoreline.geom_type.unique())}')

if shoreline.crs is None:
    raise ValueError(
        'Shapefile has no CRS. Please assign a coordinate reference system first.'
    )

# Reproject to metric CRS (UTM) if input is in degrees
if shoreline.crs.is_geographic:
    # Auto-select UTM zone from centroid longitude
    centroid    = shoreline.to_crs('EPSG:4326').geometry.unary_union.centroid
    utm_zone    = int((centroid.x + 180) / 6) + 1
    hemisphere  = 'north' if centroid.y >= 0 else 'south'
    utm_proj    = f'+proj=utm +zone={utm_zone} +{hemisphere} +datum=WGS84 +units=m +no_defs'
    shoreline_m = shoreline.to_crs(utm_proj)
    print(f'\nReprojected to UTM Zone {utm_zone} {hemisphere.upper()} for metre-based buffering')
else:
    shoreline_m = shoreline.copy()
    print('\nCRS is already in metres — no reprojection needed')


# ── CELL 5: Prepare Land Mask ────────────────────────────────
from shapely.ops import unary_union

shore_union = shoreline_m.geometry.unary_union
geom_types  = set(shoreline_m.geom_type)

land_mask = None

if LAND_POLYGON_SHP and os.path.exists(LAND_POLYGON_SHP):
    land_gdf  = gpd.read_file(LAND_POLYGON_SHP).to_crs(shoreline_m.crs)
    land_mask = land_gdf.geometry.unary_union
    print('Land polygon loaded — buffers will be clipped to land side')

elif any(t in geom_types for t in ['Polygon', 'MultiPolygon']):
    land_mask = shore_union
    print('Shoreline is a polygon — using it as the land mask')

else:
    print('NOTE: Shoreline is a line and no land polygon was provided.')
    print('Buffers will be created on BOTH sides of the line.')
    print('To restrict to land side only, set LAND_POLYGON_SHP above.')


# ── CELL 6: Generate Buffer Bands ───────────────────────────
print('\nGenerating buffer bands...')

results     = []
prev_buffer = None

for i, dist in enumerate(BUFFER_DISTANCES):

    # Full circular buffer at this distance
    full_buf = shore_union.buffer(dist)

    # Clip to land side if mask is available
    if land_mask is not None:
        land_buf = full_buf.intersection(land_mask)
    else:
        land_buf = full_buf

    # Subtract previous buffer to get the ring/band only
    if prev_buffer is not None:
        inner_dist = BUFFER_DISTANCES[i - 1]
        ring       = land_buf.difference(prev_buffer)
        band_label = f'{inner_dist}m_to_{dist}m'
    else:
        ring       = land_buf
        band_label = f'0m_to_{dist}m'

    results.append({
        'dist'      : dist,
        'label'     : band_label,
        'geometry'  : ring,
        'cumulative': land_buf
    })

    prev_buffer = land_buf
    print(f'  Band {band_label} — done')

print('\nAll 4 buffer bands created!')


# ── CELL 7: Save Shapefiles to Google Drive ─────────────────
print('\nSaving shapefiles...')

saved = []

for r in results:
    gdf = gpd.GeoDataFrame(
        {
            'buffer_m' : [r['dist']],
            'label'    : [r['label']],
            'side'     : ['land'],
        },
        geometry=[r['geometry']],
        crs=shoreline_m.crs
    )

    # Reproject back to the original CRS of the input
    gdf_out  = gdf.to_crs(shoreline.crs)
    filename = f'buffer_{r["label"]}.shp'
    out_path = os.path.join(OUTPUT_FOLDER, filename)
    gdf_out.to_file(out_path)
    saved.append(out_path)
    print(f'  Saved: {filename}')

print(f'\nDone! {len(saved)} shapefiles saved to:\n  {OUTPUT_FOLDER}')


# ── CELL 8: Plot & Save Map ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

colors = ['#cce5ff', '#74b9ff', '#0984e3', '#023e8a']
fig, ax = plt.subplots(figsize=(13, 11))

# Draw from outermost inward so inner bands appear on top
for r, color in zip(reversed(results), reversed(colors)):
    gpd.GeoDataFrame(
        geometry=[r['geometry']], crs=shoreline_m.crs
    ).plot(ax=ax, color=color, edgecolor='white', linewidth=0.6, alpha=0.9)

# Draw shoreline on top
shoreline_m.plot(ax=ax, color='#d63031', linewidth=1.8, zorder=5)

# Legend
patches = [mpatches.Patch(color=c, label=r['label']) for c, r in zip(colors, results)]
patches.append(mpatches.Patch(color='#d63031', label='Shoreline'))
ax.legend(handles=patches, loc='lower right', fontsize=10, title='Buffer Zones', title_fontsize=11)

ax.set_title('Shoreline Land-Side Buffer Zones — 500m Intervals', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Easting (m)', fontsize=11)
ax.set_ylabel('Northing (m)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

map_path = os.path.join(OUTPUT_FOLDER, 'buffer_map.png')
plt.savefig(map_path, dpi=150, bbox_inches='tight')
print(f'Map saved: {map_path}')
plt.show()


# ── CELL 9: Summary ──────────────────────────────────────────
print('=' * 52)
print('  SUMMARY')
print('=' * 52)
print(f'  Input CRS  : {shoreline.crs}')
print(f'  Buffer side: Land only')
print()
for r in results:
    try:
        area_km2 = r['geometry'].area / 1e6
    except Exception:
        area_km2 = 0.0
    print(f'  {r["label"]:25s}  ~{area_km2:,.2f} km2')
print()
print(f'  Output: {OUTPUT_FOLDER}')
print('=' * 52)
